# 第7章：模型体系架构

## 本章学习目标

- 理解 qlib 模型设计模式
- 掌握模型训练与预测接口
- 学会模型持久化
- 能够实现自定义模型

---

## 7.1 模型体系概述

Qlib 的模型系统采用模块化设计，支持多种机器学习和深度学习模型。

### 架构图

```
┌─────────────────────────────────────────────────────────────┐
│                    Qlib 模型体系                            │
├─────────────────────────────────────────────────────────────┤
│  ┌─────────────────────────────────────────────────────┐   │
│  │              BaseModel (基类)                        │   │
│  └─────────────────────────────────────────────────────┘   │
│                          ↓                                  │
│  ┌────────────┐  ┌────────────┐  ┌────────────┐           │
│  │   Model    │  │  ModelFT   │  │  其他基类  │           │
│  │  (标准)    │  │(微调支持)  │  │            │           │
│  └────────────┘  └────────────┘  └────────────┘           │
│        ↓               ↓               ↓                   │
│  ┌──────────────────────────────────────────────────┐     │
│  │ LGBModel, XGBModel, LSTM, Transformer, ...       │     │
│  └──────────────────────────────────────────────────┘     │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
import qlib
from qlib.model.base import Model, BaseModel
from qlib.workflow import R
import pandas as pd
import numpy as np

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 7.2 模型基类详解

### 7.2.1 BaseModel 类

In [ ]:
# 查看 Model 基类
from qlib.model.base import Model
import inspect

print("Model 基类定义:")
print("=" * 60)

# 查看源码
print(inspect.getsource(Model))

In [ ]:
# 查看内置模型
from qlib.contrib.model import gbdt, pytorch_lstm, pytorch_gru, pytorch_transformer

print("Qlib 内置模型:")
print("=" * 60)

# GBDT 模型
print("\nGBDT 模型:")
print(f"  - LGBModel: {gbdt.LGBModel}")

# PyTorch 模型
print("\nPyTorch 模型:")
print(f"  - LSTM: {pytorch_lstm.LSTM}")
print(f"  - GRU: {pytorch_gru.GRU}")
print(f"  - Transformer: {pytorch_transformer.Transformer}")

### 7.2.2 模型接口规范

In [ ]:
# 模型必须实现的核心方法
print("模型核心接口:")
print("=" * 60)

interface_methods = [
    ("fit(dataset)", "训练模型，接收 Dataset 对象"),
    ("predict(dataset)", "预测，返回预测结果 Series"),
    ("save(path)", "保存模型到指定路径"),
    ("load(path)", "从路径加载模型"),
]

for method, desc in interface_methods:
    print(f"  {method:25s} - {desc}")

## 7.3 实现自定义模型

### 7.3.1 简单线性模型

In [ ]:
from qlib.model.base import Model
from sklearn.linear_model import LinearRegression
import pickle

class SimpleLinearModel(Model):
    """简单的线性回归模型"""
    
    def __init__(self, **kwargs):
        self.model = None
        self.params = kwargs
    
    def fit(self, dataset):
        """训练模型"""
        # 获取训练数据
        df_train = dataset.prepare("train")
        
        # 提取特征和标签
        X = df_train['feature'].values
        y = df_train['label'].values.ravel()
        
        # 训练模型
        self.model = LinearRegression(**self.params)
        self.model.fit(X, y)
        
        return self
    
    def predict(self, dataset):
        """预测"""
        df_test = dataset.prepare("test")
        X = df_test['feature'].values
        
        pred = self.model.predict(X)
        
        # 返回带索引的 Series
        return pd.Series(pred, index=df_test.index)
    
    def save(self, path):
        """保存模型"""
        with open(path, "wb") as f:
            pickle.dump(self.model, f)
    
    def load(self, path):
        """加载模型"""
        with open(path, "rb") as f:
            self.model = pickle.load(f)
        return self

print("SimpleLinearModel 定义完成")

In [ ]:
# 测试自定义模型
from qlib.data.dataset import DatasetH
from qlib.contrib.data.handler import Alpha158

# 创建数据集
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "module_path": "qlib.contrib.data.handler",
        "kwargs": {
            "start_time": "2018-01-01",
            "end_time": "2022-12-31",
            "fit_start_time": "2018-01-01",
            "fit_end_time": "2020-12-31",
            "instruments": "csi300",
        },
    },
    segments={
        "train": ("2018-01-01", "2020-12-31"),
        "test": ("2021-01-01", "2022-12-31"),
    },
)

print("数据集创建完成")

In [ ]:
# 训练模型
model = SimpleLinearModel()
model.fit(dataset)

print("模型训练完成")

In [ ]:
# 预测
predictions = model.predict(dataset)

print(f"预测结果形状: {predictions.shape}")
print(f"\n预测结果示例:")
predictions.head()

In [ ]:
# 保存模型
from pathlib import Path

model_path = Path("./tutorials/simple_linear_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)

model.save(model_path)
print(f"模型已保存到: {model_path}")

# 加载模型
model_loaded = SimpleLinearModel()
model_loaded.load(model_path)
print("模型加载成功")

## 7.4 使用 Recorder 管理实验

In [ ]:
from qlib.workflow import R

# 使用 Recorder 记录实验
with R.start(experiment_name="linear_model_exp") as recorder:
    
    # 记录参数
    recorder.log_params({
        "model_type": "LinearRegression",
        "features": "Alpha158",
        "train_period": "2018-2020",
        "test_period": "2021-2022",
    })
    
    # 训练模型
    model = SimpleLinearModel()
    model.fit(dataset)
    
    # 预测
    predictions = model.predict(dataset)
    
    # 计算评估指标
    # IC (Information Coefficient)
    df_test = dataset.prepare("test")
    labels = df_test['label'].values.ravel()
    
    # 计算 IC
    ic = np.corrcoef(predictions.values, labels)[0, 1]
    
    # 记录指标
    recorder.log_metrics({
        "ic": ic,
        "num_predictions": len(predictions),
    })
    
    # 保存模型
    recorder.save_object(model, name="model.pkl")
    
    print(f"IC: {ic:.4f}")
    print(f"Recorder ID: {recorder.id}")

In [ ]:
# 查看 Recorder 记录
recorders = R.list_recorders(experiment_name="linear_model_exp")

for rid, rec in recorders.items():
    print(f"\nRecorder ID: {rid}")
    print(f"状态: {rec.status}")
    
    # 查看记录的参数
    params = rec.load_object("params")
    if params:
        print("参数:")
        for k, v in params.items():
            print(f"  {k}: {v}")
    
    # 查看记录的指标
    metrics = rec.load_object("metrics")
    if metrics:
        print("指标:")
        for k, v in metrics.items():
            print(f"  {k}: {v}")

## 7.5 模型评估方法

In [ ]:
# 定义评估函数
def evaluate_predictions(predictions, labels):
    """评估预测结果"""
    
    # 确保输入是一维数组
    pred = np.array(predictions).ravel()
    label = np.array(labels).ravel()
    
    # 移除 NaN
    mask = ~(np.isnan(pred) | np.isnan(label))
    pred = pred[mask]
    label = label[mask]
    
    # IC (Information Coefficient)
    ic = np.corrcoef(pred, label)[0, 1]
    
    # Rank IC
    rank_ic = np.corrcoef(np.argsort(np.argsort(pred)), np.argsort(np.argsort(label)))[0, 1]
    
    # MSE
    mse = np.mean((pred - label) ** 2)
    
    # MAE
    mae = np.mean(np.abs(pred - label))
    
    return {
        "IC": ic,
        "Rank IC": rank_ic,
        "MSE": mse,
        "MAE": mae,
        "样本数": len(pred),
    }

# 评估模型
df_test = dataset.prepare("test")
labels = df_test['label']

metrics = evaluate_predictions(predictions, labels)

print("模型评估结果:")
print("=" * 40)
for k, v in metrics.items():
    print(f"{k:15s}: {v:.4f}")

## 7.6 更复杂的自定义模型

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

class SklearnModel(Model):
    """通用的 sklearn 模型封装"""
    
    def __init__(self, model_class, model_params=None, normalize=True):
        self.model_class = model_class
        self.model_params = model_params or {}
        self.normalize = normalize
        self.model = None
        self.scaler = None
    
    def fit(self, dataset):
        """训练模型"""
        df_train = dataset.prepare("train")
        X = df_train['feature'].values
        y = df_train['label'].values.ravel()
        
        # 处理缺失值
        mask = ~np.isnan(y)
        X = X[mask]
        y = y[mask]
        
        # 标准化
        if self.normalize:
            self.scaler = StandardScaler()
            X = self.scaler.fit_transform(X)
        
        # 训练模型
        self.model = self.model_class(**self.model_params)
        self.model.fit(X, y)
        
        return self
    
    def predict(self, dataset):
        """预测"""
        df_test = dataset.prepare("test")
        X = df_test['feature'].values
        
        # 标准化
        if self.scaler is not None:
            X = self.scaler.transform(X)
        
        pred = self.model.predict(X)
        return pd.Series(pred, index=df_test.index)
    
    def save(self, path):
        """保存模型"""
        import pickle
        with open(path, "wb") as f:
            pickle.dump({"model": self.model, "scaler": self.scaler}, f)
    
    def load(self, path):
        """加载模型"""
        import pickle
        with open(path, "rb") as f:
            data = pickle.load(f)
            self.model = data["model"]
            self.scaler = data.get("scaler")
        return self

print("SklearnModel 定义完成")

In [ ]:
# 测试随机森林模型
rf_model = SklearnModel(
    model_class=RandomForestRegressor,
    model_params={
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42,
        "n_jobs": -1,
    },
    normalize=True,
)

print("开始训练随机森林模型...")
rf_model.fit(dataset)
print("训练完成")

In [ ]:
# 预测并评估
rf_predictions = rf_model.predict(dataset)
rf_metrics = evaluate_predictions(rf_predictions, labels)

print("随机森林模型评估结果:")
print("=" * 40)
for k, v in rf_metrics.items():
    print(f"{k:15s}: {v:.4f}")

In [ ]:
# 对比不同模型
comparison = pd.DataFrame({
    "线性回归": metrics,
    "随机森林": rf_metrics,
}).T

print("模型对比:")
comparison

## 7.7 实践练习

In [ ]:
# 练习1: 实现一个 Ridge 回归模型
# 并与线性回归对比效果

# 你的代码



# 参考答案
# from sklearn.linear_model import Ridge
# class RidgeModel(SklearnModel):
#     def __init__(self, alpha=1.0, **kwargs):
#         super().__init__(
#             model_class=Ridge,
#             model_params={"alpha": alpha},
#             **kwargs
#         )

In [ ]:
# 练习2: 实现一个支持早停的模型
# 当验证集性能不再提升时停止训练

# 你的代码



# 提示：可以使用 sklearn 的 partial_fit 或自定义训练循环

In [ ]:
# 练习3: 使用 Recorder 管理多个模型的实验
# 比较不同参数设置的效果

# 你的代码



# 参考答案
# with R.start(experiment_name="model_comparison"):
#     for max_depth in [5, 10, 15]:
#         model = SklearnModel(
#             RandomForestRegressor,
#             model_params={"max_depth": max_depth, "n_estimators": 100}
#         )
#         model.fit(dataset)
#         pred = model.predict(dataset)
#         metrics = evaluate_predictions(pred, labels)
#         R.log_metrics({f"depth_{max_depth}_ic": metrics["IC"]})

## 7.8 本章小结

本章我们学习了：

1. **模型体系架构**：
   - `Model` 基类接口
   - 内置模型类型

2. **自定义模型开发**：
   - 继承 `Model` 基类
   - 实现 `fit()` 和 `predict()` 方法
   - 模型保存与加载

3. **实验管理**：
   - 使用 Recorder 记录实验
   - 参数和指标日志
   - 模型持久化

4. **模型评估**：
   - IC / Rank IC
   - MSE / MAE

### 关键 API 速查

```python
# 定义模型
class MyModel(Model):
    def fit(self, dataset): ...
    def predict(self, dataset): ...
    def save(self, path): ...
    def load(self, path): ...

# 使用 Recorder
with R.start(experiment_name="my_exp") as recorder:
    recorder.log_params({...})
    recorder.log_metrics({...})
    recorder.save_object(model, name="model.pkl")
```

### 下一章预告

下一章我们将深入学习传统机器学习模型，包括：
- LightGBM 模型配置与训练
- XGBoost 模型使用
- 超参数调优